In [1]:
# Cell 1 - Imports
import requests
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Cell 2 - Load existing data (1920-2021)
df = pd.read_csv('merged_article_counts_by_country_across_years.csv')
print(f"Loaded: {len(df)} countries, years 1920-2021")


Loaded: 197 countries, years 1920-2021


In [ ]:
# Cell 3 - Fetch article counts for 2022-2025 & update totals to 1920-2025
api_key = 'eb2cec89a26c7449245d9379a4f9944e'
base_url = 'https://api.elsevier.com/content/search/scopus'
headers = {'Accept': 'application/json'}

# 6 countries where Scopus uses a different name than the backbone
scopus_name_map = {
    'United States of America': 'United States',
    'Hong Kong S.A.R.': 'Hong Kong',
    'Vietnam': 'Viet Nam',
    'Syria': 'Syrian Arab Republic',
    'Ivory Coast': "Cote d'Ivoire",
    'Democratic Republic of the Congo': 'Democratic Republic Congo',
    'Russia': 'Russian Federation',
}

for i in range(len(df)):
    country = df.loc[i, 'country']
    if pd.isnull(country):
        continue

    # Use Scopus-compatible name (if necessary) 
    query_country = scopus_name_map.get(country, country)

    print(f"[{i+1}/{len(df)}] {country}...", end=" ", flush=True)

    for year in range(2022, 2026):
        query = f'TITLE-ABS-KEY("anesthes*" OR "anaesthes*") AND AFFILCOUNTRY({query_country})'
        params = {'query': query, 'count': 25, 'date': str(year), 'apiKey': api_key}
        response = requests.get(base_url, headers=headers, params=params)
        if response.status_code == 200:
            data = response.json()
            total = data.get('search-results', {}).get('opensearch:totalResults', 0)
            df.loc[i, str(year)] = int(total)
        time.sleep(1)

    # Update total count (1920-2025)
    query = f'TITLE-ABS-KEY("anesthes*" OR "anaesthes*") AND AFFILCOUNTRY({query_country})'
    params = {'query': query, 'count': 25, 'date': '1920-2025', 'apiKey': api_key}
    response = requests.get(base_url, headers=headers, params=params)
    if response.status_code == 200:
        data = response.json()
        total = data.get('search-results', {}).get('opensearch:totalResults', 0)
        df.loc[i, 'anesthesiology'] = int(total)

    print("done.")

In [4]:
# Cell 4 - Update HDI to 2023 values from 2025 HDR
hdi_new = pd.read_csv('backbone_anesthesiology_2023hdi.csv')
hdi_lookup = hdi_new.set_index('country')[['hdi_2023', 'hdicode']].to_dict('index')

for i, row in df.iterrows():
    if row['country'] in hdi_lookup:
        info = hdi_lookup[row['country']]
        df.loc[i, 'hdi_2023'] = info['hdi_2023']
        df.loc[i, 'hdicode'] = info['hdicode']

if 'hdi_2022' in df.columns:
    df = df.drop(columns=['hdi_2022'])

In [5]:
# Cell 5 - Save
df.to_csv('UPDATED_merged_article_counts_by_country_across_years.csv', index=False)
print(f"Saved! Shape: {df.shape}")
print(df[['country', 'hdicode', 'hdi_2023', 'anesthesiology', '2022', '2023', '2024', '2025']].head(10))



Saved! Shape: (197, 112)
               country    hdicode  hdi_2023  anesthesiology   2022   2023  \
0          Afghanistan        Low     0.496              49    4.0    7.0   
1              Albania  Very High     0.810              63    5.0    5.0   
2              Algeria       High     0.763             103    2.0   11.0   
3              Andorra  Very High     0.913               4    0.0    1.0   
4               Angola     Medium     0.616              11    0.0    0.0   
5  Antigua and Barbuda  Very High     0.851              10    1.0    1.0   
6            Argentina  Very High     0.865            1289   86.0   74.0   
7              Armenia  Very High     0.811              65    9.0   10.0   
8            Australia  Very High     0.958           13559  640.0  539.0   
9              Austria  Very High     0.930            4832  234.0  211.0   

    2024   2025  
0    3.0    9.0  
1   11.0   10.0  
2   19.0   18.0  
3    1.0    0.0  
4    1.0    3.0  
5    2.0    1.0  
6

In [ ]:
# Additional imports for analysis
from collections import defaultdict
from urllib.parse import urlparse, parse_qs
from tqdm import tqdm
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib_venn import venn2, venn2_circles
import numpy as np
import seaborn as sb
import statsmodels.api as sm
from scipy.stats import pearsonr
from scipy.stats import mannwhitneyu
from scipy.stats import kruskal

In [ ]:
#Adjudication of results from above. Script written for Sierra Leone.
api_key = 'eb2cec89a26c7449245d9379a4f9944e'
base_url = 'https://api.elsevier.com/content/search/scopus'
headers = {'Accept': 'application/json'}
query = 'TITLE-ABS-KEY("anesthes*" OR "anaesthes*") AND AFFILCOUNTRY("Sierra Leone")'
params = {
    'query': query,
    'count': 25,  # Max number of results per page
    'date': '1920-2025',
    'apiKey': api_key
}

# List to hold all the entries
all_entries = []

# Initial request
page_number = 1
while True:
    # Make the GET request
    response = requests.get(base_url, headers=headers, params=params)
    
    # Check if the response is successful
    if response.status_code == 200:
        data = response.json()
        total = data.get('search-results', {})
        entries = total.get('entry', [])
        
        # Add a tqdm progress bar for the processing of the current page's entries
        for entry in tqdm(entries, desc=f"Processing Page {page_number}", leave=False):
            title = entry.get('dc:title', "No Title")
            authors = entry.get('dc:creator', "No Author")
            publication_name = entry.get('prism:publicationName', "No Publication Name")
            publication_date = entry.get('prism:coverDate', "No Date")
            
            # Append the details to the list
            all_entries.append({
                "title": title,
                "authors": authors,
                "publication_name": publication_name,
                "publication_date": publication_date
            })
        
        # Check if there is a 'next' link for pagination
        links = total.get('link', [])
        next_link = None
        for link in links:
            if link.get('@ref') == 'next':
                next_link = link.get('@href')
                break
        
        # If there is a 'next' link, update the params for the next page
        if next_link:
            # Update params with the next page URL
            parsed_url = urlparse(next_link)
            params['start'] = parse_qs(parsed_url.query).get('start', [None])[0]
            page_number += 1  # Increment the page number for the progress bar
        else:
            # No more pages, exit the loop
            break
    else:
        # Handle the case where the request fails
        print(f"Error: {response.status_code}")
        break

# Convert the list of entries into a pandas DataFrame
df = pd.DataFrame(all_entries)

# Output the collected DataFrame
print(f"Collected {len(df)} entries.")
print(df.head())  # Print the first few rows to check the results
df.to_csv('script_adjudication_sierra_leone.csv')

In [ ]:
#Creating World Dataset from Total Article Counts
##Note Total Article Count dataset requires manual refinement
##given duplicate country names. Refinement should be completed
##prior to merging with Geopandas dataset. Countries excluded
##from refinement include Czechoslovakia, Yugoslavia, Macao, 
##Gibraltar, Reunion, Northern Mariana Islands, and No Country
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)
data_input = pd.read_csv('UPDATED_merged_article_counts_by_country_across_years.csv')
oecd = pd.read_csv('oecd_iso3_codes.csv')
world = world.merge(data_input, left_on='ADM0_A3', right_on='iso3', how='left')
world['log'] = np.log10(world['anesthesiology'].abs()+1)

In [ ]:
#Atlas Generation on World Dataset
fig, ax = plt.subplots(1, 1, figsize=(15, 10))
world.boundary.plot(ax=ax, edgecolor='black', linewidth=0.5,)  # Plot the boundaries of countries
plot = world.plot(column='log', ax=ax, legend=True,
           legend_kwds={'label': "Log10 Normalized National Publication Volume",
                        'orientation': "horizontal"},
           cmap='Blues').set_axis_off()  # You can change the color map (e.g., 'Oranges', 'Blues', 'YlGnBu')
#plt.savefig("National Publication Density.pdf", format="pdf")
plt.show()

In [ ]:
#Analysis on World dataset
sb.set(style = 'white')
desired_order = ['Very High', 'High', 'Medium', 'Low']
sb.violinplot(x ="hdicode", y = np.log10(data_input['anesthesiology'].abs()+1), 
              order = desired_order, palette = "PuBu_r",
              data = data_input)
#plt.savefig("HDI Distribution.pdf", format="pdf")

In [ ]:
#Bland-Altman Plot
f, ax = plt.subplots(1, figsize = (8,5))
sm.graphics.mean_diff_plot(np.log10(data_input['anesthesiology'].abs()+1), np.log10(data_input['total_yearly'].abs()+1), ax = ax)
ax.set_ylim(-1, 1)
#plt.savefig("Bland-Altman.pdf", format="pdf")
plt.show()

In [ ]:
#Statistical Analyses
print(data_input['hdicode'].value_counts())

hdi_very_high = data_input.loc[data_input["hdicode"]=="Very High", ]
hdi_high = data_input.loc[data_input["hdicode"]=="High", ]
hdi_medium = data_input.loc[data_input["hdicode"]=="Medium", ]
hdi_low = data_input.loc[data_input["hdicode"]=="Low", ]
top_performers = data_input.sort_values(by='anesthesiology', ascending=False).head(10)
oecd_performers = oecd.merge(data_input, left_on='ISO3', right_on='iso3', how='left').drop(['Country','ISO3'], axis=1)

stat, p_value = kruskal(np.log10(hdi_very_high['anesthesiology'].abs()+1),
                        np.log10(hdi_high['anesthesiology'].abs()+1),
                        np.log10(hdi_medium['anesthesiology'].abs()+1),
                        np.log10(hdi_low['anesthesiology'].abs()+1))

print("Kruskal-Wallis statistic:", stat)
print("p-value:", p_value)

In [ ]:
#Venn Diagram of Top Performers and OECD
# Extract sets from the 'Name' column
set1 = set(top_performers['iso3'])
set2 = set(oecd_performers['iso3'])

# Create the figure and axes
fig, ax = plt.subplots(figsize=(8, 6))
venn = venn2([set1, set2], set_labels=('Top Performers', 'OECD'), ax=ax, set_colors=('#f7fbff', '#08306b'), alpha=0.7)

# Add circle styling
venn2_circles([set1, set2], linestyle='solid', linewidth=1.5, color='black')

# Custom label styling
for text in venn.set_labels:
    text.set_fontsize(14)
    text.set_fontweight('bold')

for text in venn.subset_labels:
    if text:
        text.set_fontsize(12)

# Add title with custom font
plt.title('Overlap Between Top Performers and OECD', fontsize=16, fontweight='bold')

# Remove spines and ticks for a cleaner look
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_xticks([])
ax.set_yticks([])

# Save if needed
#plt.savefig("Venn.pdf", format="pdf")

#Display
plt.tight_layout()
plt.show()
only_top = sorted(set1 - set2)
only_oecd = sorted(set2 - set1)
overlap = sorted(set1 & set2)

# Create table DataFrame
venn_table = pd.DataFrame({
    'Only in Top Performers': pd.Series(only_top),
    'Overlap': pd.Series(overlap),
    'Only in OECD': pd.Series(only_oecd)
})

print(venn_table)

In [ ]:
#Scatterplot
sb.scatterplot(data=data_input, x=np.log10(data_input['anesthesiology'].abs()+1), y=np.log10(data_input['total_yearly'].abs()+1))
corr_coef, p_value = pearsonr(np.log10(data_input['anesthesiology'].abs()+1), np.log10(data_input['total_yearly'].abs()+1))
m, b = np.polyfit(np.log10(data_input['anesthesiology'].abs()+1), np.log10(data_input['total_yearly'].abs()+1), 1)  # 1 is the degree of the polynomial (line)
plt.plot(np.log10(data_input['anesthesiology'].abs()+1), m*np.array(np.log10(data_input['anesthesiology'].abs()+1)) + b, color='black')
plt.title(f'Scatter plot\nCorrelation: {corr_coef:.2f}, p-value: {p_value:.3f}')
#plt.savefig("Scatterplot.pdf", format="pdf")
plt.show()

In [ ]:
#Time Trend for Total Article Counts
start_year = '1920'
end_year = '2025'

# Select a range of columns (inclusive of start and end year)
df_range = data_input.loc[:, start_year:end_year]
column_sums = df_range.sum(axis=0)
plt.figure(figsize=(10, 6))
plt.plot(column_sums.index, column_sums.values, color='navy', label='Sum of Values')

# Shade the area under the curve
plt.fill_between(column_sums.index, column_sums.values, color='steelblue', alpha=0.4)

# Adding labels and title
plt.title('Sum of Columns over Time')
plt.xlabel('Year')
plt.ylabel('Sum of Values')
plt.grid(False)

# Customizing the x-axis: Display only every 10th year
# Generate a list of every 10th year (e.g., from 2000, 2010, etc.)
xticks = [year for i, year in enumerate(column_sums.index) if i % 10 == 0]

# Set the x-axis ticks and labels
plt.xticks(xticks)

# Show the plot
#plt.savefig("Time Series.pdf", format="pdf")
plt.show()

In [ ]:
#Time Trend for Total Article Counts by HDI
start_year = '1960'
end_year = '2025'
plt.figure(figsize=(10, 6))

hdi_codes = ['Very High', 'High', 'Medium', 'Low']

# Get a color palette from the "PuBu_r" colormap (reverse version of the "PuBu" palette)
colors = plt.cm.jet(np.linspace(0, 1, len(hdi_codes)))

# Select a range of columns (inclusive of start and end year)
for i, code in enumerate(hdi_codes):
    df_range = data_input.loc[data_input["hdicode"]==code, start_year:end_year]
    column_sums = df_range.sum(axis=0)
    plt.plot(column_sums.index, column_sums.values, label=code, color=colors[i])
    plt.fill_between(column_sums.index, column_sums.values, color=colors[i], alpha=0.4)
    
# Adding labels and title
plt.title('Sum of Columns over Time')
plt.xlabel('Year')
plt.ylabel('Sum of Values')
plt.grid(False)

# Customizing the x-axis: Display only every 10th year
# Generate a list of every 10th year (e.g., from 2000, 2010, etc.)
xticks = [year for i, year in enumerate(column_sums.index) if i % 10 == 0]

# Set the x-axis ticks and labels
plt.xticks(xticks)

# Add a legend to distinguish the lines by HDI code
plt.legend(title='HDI Code')

# Show the plot
#plt.savefig("Time Series by HDI Code.pdf", format="pdf")
plt.show()

In [ ]:
#Time Trend for Total Article Counts in Top Performers
start_year = '1960'
end_year = '2025'
plt.figure(figsize=(10, 6))

# Select a range of columns (inclusive of start and end year)
nations = top_performers["country"]

# Get a color palette from the "PuBu_r" colormap (reverse version of the "PuBu" palette)
colors = plt.cm.turbo(np.linspace(0, 1, len(top_performers['country'])))

for i, nation in enumerate(nations):
    df_range = top_performers.loc[top_performers["country"]==nation, start_year:end_year]
    column_sums = df_range.sum(axis=0)
    plt.plot(column_sums.index, column_sums.values, label=nation, color=colors[i])
    #plt.fill_between(column_sums.index, column_sums.values, color=colors[i], alpha=0.4)
    
# Adding labels and title
plt.title('Sum of Columns over Time')
plt.xlabel('Year')
plt.ylabel('Sum of Values')
plt.grid(False)

# Customizing the x-axis: Display only every 10th year
# Generate a list of every 10th year (e.g., from 2000, 2010, etc.)
xticks = [year for i, year in enumerate(column_sums.index) if i % 10 == 0]

# Set the x-axis ticks and labels
plt.xticks(xticks)

# Add a legend to distinguish the lines by HDI code
plt.legend(title='Country')

# Show the plot
#plt.savefig("Time Series by Country (Top Performers).pdf", format="pdf")
plt.show()

In [ ]:
#Long format transformtation for all article counts
data_input_clean = data_input.drop(['hdicode','anesthesiology','total_yearly'], axis=1)
data_input_clean_long = pd.melt(data_input_clean, id_vars=['country','hdi_2023','iso3'], var_name='year', value_name='count')
data_input_clean_long['year'] = data_input_clean_long['year'].astype(int)
data_input_final = data_input_clean_long[data_input_clean_long['year']>=1960].reset_index(drop=True)

In [ ]:
#Long format transformtation for top performer article counts
top_performers_clean = top_performers.drop(['hdicode','anesthesiology','total_yearly','hdi_2023'], axis=1)
top_performers_clean_long = pd.melt(top_performers_clean, id_vars=['country','iso3'], var_name='year', value_name='count')
top_performers_clean_long['year'] = top_performers_clean_long['year'].astype(int)
top_performers_final = top_performers_clean_long[top_performers_clean_long['year']>=1960].reset_index(drop=True)

In [ ]:
#Long format transformtation for OECD article counts
oecd_clean = oecd_performers.drop(['hdicode','anesthesiology','total_yearly','hdi_2023'], axis=1)
oecd_clean_long = pd.melt(oecd_clean, id_vars=['country','iso3'], var_name='year', value_name='count')
oecd_clean_long['year'] = oecd_clean_long['year'].astype(int)
oecd_final = oecd_clean_long[oecd_clean_long['year']>=1995].reset_index(drop=True)

In [ ]:
#Long format transformtation of World Bank indicators
world_bank = pd.read_csv('world_bank.csv')
world_bank_long = world_bank.melt(id_vars=['Series Code','Country Name','Country Code'], var_name='year', value_name='count')
world_bank_long = world_bank_long.drop('Country Name', axis=1)
world_bank_long['year'] = pd.to_numeric(world_bank_long['year'])
world_bank_long_pivot = world_bank_long.pivot_table(
    index=['Country Code', 'year'],
    columns='Series Code',
    values='count',
    aggfunc='sum'  # or 'mean', etc., depending on your use case
).reset_index()
world_bank_long_pivot = world_bank_long_pivot.rename(columns={'Country Code': 'iso3'})
world_bank_long_pivot = world_bank_long_pivot.drop('GB.XPD.RSDV.GD.ZS', axis=1)

In [ ]:
#Merging Top Performer Article Counts with World Bank Indicators
pvar_dataset_top = top_performers_final.merge(world_bank_long_pivot, on = ['iso3','year'], how = 'left')
pvar_dataset_top['country'] = pvar_dataset_top['country'].astype(str)      # or .astype('category') if you like
pvar_dataset_top['year'] = pvar_dataset_top['year'].astype(int)          # or float, but must be numeric
for col in ['count', 'NY.GDP.MKTP.CD','NY.GDP.PCAP.CD']:
    pvar_dataset_top[col] = pd.to_numeric(pvar_dataset_top[col], errors='coerce')
pvar_dataset_nona = pvar_dataset_top.dropna()

In [ ]:
plt.figure(figsize=(10, 6))
scatter = sb.scatterplot(
    data=pvar_dataset_nona,
    x='year',
    y='count',
    hue='country',
    size='NY.GDP.MKTP.CD',
    sizes=(50, 500),
    palette=colors.tolist(),
    edgecolor='black',
    linewidth=0.8,
    legend=False
)

color_handles = [
    Line2D([0], [0], marker='o', color='w', label=country,
           markerfacecolor=color, markersize=10, markeredgecolor='black')
    for country, color in zip(pvar_dataset_nona['country'], sb.color_palette(colors.tolist()))
]

legend1 = plt.legend(handles=color_handles, title='Country', loc='upper left')
plt.gca().add_artist(legend1)
plt.savefig("Time Series by Country with GDP (Top Performers).pdf", format="pdf")
plt.show()

In [ ]:
#Merging OECD Article Counts with World Bank Indicators
pvar_dataset_oecd = oecd_final.merge(world_bank_long_pivot, on = ['iso3','year'], how = 'left')
pvar_dataset_oecd['country'] = pvar_dataset_oecd['country'].astype(str)      # or .astype('category') if you like
pvar_dataset_oecd['year'] = pvar_dataset_oecd['year'].astype(int)          # or float, but must be numeric
for col in ['count', 'NY.GDP.MKTP.CD','NY.GDP.PCAP.CD']:
    pvar_dataset_oecd[col] = pd.to_numeric(pvar_dataset_oecd[col], errors='coerce')
pvar_dataset_oecd_nona = pvar_dataset_oecd.dropna()

In [ ]:
#Alternative method for pulling articles per year
#Not comprehensive as it relies on query of "anesthesiology" as opposed to "anesthes*"
# List of years to process
years = range(1920, 2026)
all_country_counts = defaultdict(lambda: defaultdict(int))  # {country: {year: count}}

# Process data for each year
for year in years:
    print(f"Processing data for {year}...")
    country_counts = collect_articles_per_country(year)
    
    # Store the counts for each country and year
    for country, count in country_counts.items():
        all_country_counts[country][year] = count

# Convert the dictionary to a pandas DataFrame
df = pd.DataFrame.from_dict(all_country_counts, orient='index')
df_sorted = df.sort_index(axis=1)
df_sorted.to_csv('article_counts_by_country.csv')